# <span style="color: #1F1DB5;">SPRINT 8 – Recopilación y Almacenamiento de Datos

# <span style="color: #1F1DB5;">Notebook 2 – SQL Avanzado — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Escribir consultas SQL complejas: JOINs múltiples, subconsultas, CTEs.
- Aplicar Window Functions: ROWNUMBER, RANK, LAG, LEAD, SUM OVER.
- Usar índices y entender su impacto en el rendimiento de queries.
- Conectar Python con bases de datos SQL usando SQLite y pd.readsql().
- Aplicar buenas prácticas de SQL para eficiencia y legibilidad.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Escribir consultas SQL **complejas**: JOINs múltiples, subconsultas, CTEs
- Aplicar **Window Functions**: `ROW_NUMBER`, `RANK`, `LAG`, `LEAD`, `SUM OVER`
- Usar **índices** y entender su impacto en el rendimiento de queries
- Conectar **Python con bases de datos SQL** usando SQLite y pd.read_sql()
- Aplicar **buenas prácticas** de SQL para eficiencia y legibilidad

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Repaso de SQL y conexión con Pandas | 10 min |
JOINs múltiples y subconsultas | 25 min |
CTEs: Common Table Expressions | 15 min |
Window Functions | 25 min |
Python + SQL: queries desde código | 15 min |
Actividad práctica – Breakout Rooms | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo extraer exactamente los datos que necesitas de una base de datos con millones de registros?

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Repaso de SQL y Conexión con Pandas (10 mins)

SQL (Structured Query Language) es el lenguaje universal para interactuar con bases de datos relacionales. Como Data Scientist, lo usarás **todos los días** para extraer datos antes de analizarlos en Python.

### ¿Por qué SQL sigue siendo fundamental?

- Los datos de las empresas viven en bases de datos SQL (PostgreSQL, MySQL, BigQuery)
- Es más eficiente filtrar y agregar en la base de datos que traer todo a Python
- Las entrevistas técnicas de DS **siempre** incluyen preguntas de SQL
- Complementa a Pandas: SQL para extraer, Pandas para transformar y analizar

### Orden de ejecución (recordatorio crucial)

El orden en que **escribes** una query NO es el orden en que se **ejecuta**:

```
Orden de escritura:  SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
Orden de ejecución:  FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

**Consecuencia práctica:** No puedes usar un alias definido en `SELECT` dentro de `WHERE` (porque `WHERE` se ejecuta antes). Pero sí puedes usarlo en `ORDER BY` (se ejecuta después).

### Analogía SQL ↔ Pandas

| SQL | Pandas | Qué hace |
|---|---|---|
| `SELECT col1, col2` | `df[["col1", "col2"]]` | Seleccionar columnas |
| `WHERE condicion` | `df[df["col"] > x]` | Filtrar filas |
| `GROUP BY col` | `df.groupby("col")` | Agrupar |
| `JOIN tabla ON` | `pd.merge(df1, df2, on=)` | Combinar tablas |
| `ORDER BY col` | `df.sort_values("col")` | Ordenar |
| `LIMIT n` | `df.head(n)` | Limitar filas |

Preguntas:

- ¿Por qué es más eficiente filtrar datos en SQL que traer todo a Python y filtrar con Pandas?

- ¿Qué pasa si intentas usar un alias de SELECT en la cláusula WHERE?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">JOINs Múltiples y Subconsultas (25 mins)

En el mundo real, los datos **nunca** están en una sola tabla. Una tienda online típica tiene:
- Tabla `clientes` (quién compra)
- Tabla `productos` (qué se vende)
- Tabla `pedidos` (cuándo se compró)
- Tabla `detalle_pedidos` (qué productos tiene cada pedido)

Para responder preguntas de negocio como "¿Cuál es el producto más comprado por clientes de CDMX?" necesitas **combinar** información de múltiples tablas. Eso es exactamente lo que hacen los JOINs.

### Tipos de JOIN

| Tipo | Qué devuelve | Cuándo usarlo |
|---|---|---|
| `INNER JOIN` | Solo filas con coincidencia en ambas tablas | Cuando necesitas datos completos |
| `LEFT JOIN` | Todas las filas de la izquierda + coincidencias | Cuando quieres conservar todos los registros principales |
| `RIGHT JOIN` | Todas las filas de la derecha + coincidencias | Poco usado (invierte el LEFT) |
| `FULL OUTER JOIN` | Todas las filas de ambas tablas | Cuando necesitas ver qué NO tiene coincidencia |
| `CROSS JOIN` | Producto cartesiano (todas × todas) | Generar combinaciones |

### JOINs múltiples (3+ tablas)

Cuando necesitas datos de 3 o más tablas, simplemente encadenas JOINs. Piensa en ello como ir conectando tablas una por una, siempre a través de una columna en común.

In [ ]:
import sqlite3
import pandas as pd

# === CREAR BASE DE DATOS DE EJEMPLO: E-COMMERCE ===
conn = sqlite3.connect(":memory:")

conn.executescript('''
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY,
        nombre TEXT,
        ciudad TEXT,
        fecha_registro TEXT
    );
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY,
        nombre TEXT,
        categoria TEXT,
        precio REAL
    );
    CREATE TABLE pedidos (
        id INTEGER PRIMARY KEY,
        cliente_id INTEGER,
        fecha TEXT,
        total REAL
    );
    CREATE TABLE detalle_pedidos (
        id INTEGER PRIMARY KEY,
        pedido_id INTEGER,
        producto_id INTEGER,
        cantidad INTEGER
    );

    INSERT INTO clientes VALUES
        (1,"Ana García","CDMX","2023-01-15"),
        (2,"Luis Pérez","GDL","2023-03-20"),
        (3,"María López","MTY","2023-02-10"),
        (4,"Carlos Ruiz","CDMX","2023-06-01"),
        (5,"Sofía Torres","GDL","2023-04-15");

    INSERT INTO productos VALUES
        (1,"Laptop Pro","Electrónica",15000),
        (2,"Mouse Ergonómico","Electrónica",500),
        (3,"Libro: Python para DS","Libros",350),
        (4,"Teclado Mecánico","Electrónica",800),
        (5,"Curso DS Online","Educación",5000);

    INSERT INTO pedidos VALUES
        (1,1,"2024-01-10",15500),(2,1,"2024-02-15",850),
        (3,2,"2024-01-20",350),(4,3,"2024-03-01",20000),
        (5,1,"2024-03-10",5000),(6,4,"2024-02-28",500),
        (7,2,"2024-03-15",5800),(8,5,"2024-01-25",15350);

    INSERT INTO detalle_pedidos VALUES
        (1,1,1,1),(2,1,2,1),(3,2,2,1),(4,2,3,1),
        (5,3,3,1),(6,4,1,1),(7,4,5,1),(8,5,5,1),
        (9,6,2,1),(10,7,4,1),(11,7,5,1),(12,8,1,1),(13,8,3,1);
''')
print('✅ Base de datos de e-commerce creada')
print('   Tablas: clientes, productos, pedidos, detalle_pedidos')

Ahora veamos cómo combinar 3 tablas para responder: **¿Qué productos compró cada cliente?**

In [ ]:
# JOIN de 3 tablas: clientes + pedidos + detalle + productos
query_join = '''
SELECT
    c.nombre AS cliente,
    c.ciudad,
    p.nombre AS producto,
    p.categoria,
    p.precio,
    pe.fecha AS fecha_compra
FROM detalle_pedidos d
JOIN pedidos pe ON d.pedido_id = pe.id          -- conectar detalle con pedido
JOIN clientes c ON pe.cliente_id = c.id         -- conectar pedido con cliente
JOIN productos p ON d.producto_id = p.id        -- conectar detalle con producto
ORDER BY c.nombre, pe.fecha
'''

print('¿Qué compró cada cliente?')
print(pd.read_sql(query_join, conn).to_string(index=False))

Preguntas:

- ¿Por qué necesitamos la tabla `detalle_pedidos` como puente entre pedidos y productos?

- ¿Qué pasaría si usamos LEFT JOIN en lugar de INNER JOIN en esta query?

- ¿Cómo modificarías la query para ver solo compras de clientes de CDMX?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">CTEs: Common Table Expressions (15 mins)

Una **CTE** (Common Table Expression) es una tabla temporal que defines al inicio de tu query usando la cláusula `WITH`. Es como crear una variable que contiene un resultado intermedio.

### ¿Por qué usar CTEs?

**Sin CTE** (subconsulta anidada — difícil de leer):
```sql
SELECT * FROM (
    SELECT cliente_id, SUM(total) as gasto
    FROM (
        SELECT * FROM pedidos WHERE fecha > "2024-01-01"
    ) sub1
    GROUP BY cliente_id
) sub2
WHERE gasto > 1000
```

**Con CTE** (legible y mantenible):
```sql
WITH pedidos_recientes AS (
    SELECT * FROM pedidos WHERE fecha > "2024-01-01"
),
gasto_por_cliente AS (
    SELECT cliente_id, SUM(total) as gasto
    FROM pedidos_recientes
    GROUP BY cliente_id
)
SELECT * FROM gasto_por_cliente WHERE gasto > 1000
```

### Ventajas de las CTEs
- **Legibilidad** — cada paso tiene un nombre descriptivo
- **Reutilización** — puedes referenciar la misma CTE múltiples veces
- **Debugging** — puedes ejecutar cada CTE por separado para verificar
- **Mantenimiento** — es fácil modificar un paso sin romper los demás

In [ ]:
# === CTE: Resumen de clientes con ranking ===
query_cte = '''
WITH resumen_cliente AS (
    -- Paso 1: Calcular métricas por cliente
    SELECT
        cliente_id,
        COUNT(*) as num_pedidos,
        SUM(total) as total_gastado,
        AVG(total) as ticket_promedio,
        MAX(fecha) as ultima_compra
    FROM pedidos
    GROUP BY cliente_id
)
-- Paso 2: Unir con datos del cliente y ordenar
SELECT
    c.nombre,
    c.ciudad,
    r.num_pedidos,
    r.total_gastado,
    ROUND(r.ticket_promedio, 0) as ticket_promedio,
    r.ultima_compra
FROM clientes c
JOIN resumen_cliente r ON c.id = r.cliente_id
ORDER BY r.total_gastado DESC
'''

print('Resumen de clientes (usando CTE):')
print(pd.read_sql(query_cte, conn).to_string(index=False))

Preguntas:

- ¿Cuándo usarías una CTE en lugar de una subconsulta anidada?

- ¿Puedes tener múltiples CTEs en una misma query? ¿Cómo?

- ¿Cuál es la diferencia entre una CTE y una tabla temporal (`CREATE TEMP TABLE`)?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Window Functions (25 mins)

Las **Window Functions** son una de las herramientas más poderosas de SQL moderno. Permiten hacer cálculos sobre un conjunto de filas **sin colapsar** los resultados (a diferencia de `GROUP BY`, que reduce filas).

### ¿Cuál es la diferencia con GROUP BY?

| Aspecto | GROUP BY | Window Function |
|---|---|---|
| Resultado | Una fila por grupo | Mantiene todas las filas |
| Información | Solo el agregado | Agregado + detalle original |
| Uso | Resúmenes | Rankings, acumulados, comparaciones |

### Sintaxis general
```sql
funcion() OVER (
    PARTITION BY columna_grupo    -- opcional: divide en grupos
    ORDER BY columna_orden        -- opcional: define el orden
)
```

### Funciones más usadas

| Función | Qué hace | Caso de uso en DS |
|---|---|---|
| `ROW_NUMBER()` | Número secuencial único | Seleccionar el registro más reciente por grupo |
| `RANK()` | Ranking (con empates) | Top N productos por categoría |
| `DENSE_RANK()` | Ranking sin saltos | Rankings sin huecos |
| `LAG(col, n)` | Valor de n filas atrás | Comparar con el período anterior |
| `LEAD(col, n)` | Valor de n filas adelante | Predecir el siguiente valor |
| `SUM() OVER()` | Suma acumulada | Ingresos acumulados en el tiempo |
| `AVG() OVER()` | Promedio móvil | Suavizar series temporales |
| `NTILE(n)` | Divide en n grupos iguales | Crear percentiles, cuartiles |

In [ ]:
# === WINDOW FUNCTIONS EN ACCIÓN ===

# 1. RANKING: ¿Quién es el mejor cliente por ciudad?
query_rank = '''
WITH gasto AS (
    SELECT cliente_id, SUM(total) as total_gastado
    FROM pedidos GROUP BY cliente_id
)
SELECT
    c.nombre,
    c.ciudad,
    g.total_gastado,
    RANK() OVER (PARTITION BY c.ciudad ORDER BY g.total_gastado DESC) as ranking_ciudad
FROM clientes c
JOIN gasto g ON c.id = g.cliente_id
'''
print('1. Ranking de clientes por ciudad:')
print(pd.read_sql(query_rank, conn).to_string(index=False))

In [ ]:
# 2. LAG: Comparar cada pedido con el anterior del mismo cliente
query_lag = '''
SELECT
    c.nombre,
    p.fecha,
    p.total,
    LAG(p.total, 1) OVER (PARTITION BY p.cliente_id ORDER BY p.fecha) as pedido_anterior,
    p.total - LAG(p.total, 1) OVER (PARTITION BY p.cliente_id ORDER BY p.fecha) as diferencia
FROM pedidos p
JOIN clientes c ON p.cliente_id = c.id
ORDER BY c.nombre, p.fecha
'''
print('\n2. Comparación con pedido anterior (LAG):')
print(pd.read_sql(query_lag, conn).to_string(index=False))

In [ ]:
# 3. SUM OVER: Gasto acumulado por cliente
query_acum = '''
SELECT
    c.nombre,
    p.fecha,
    p.total,
    SUM(p.total) OVER (PARTITION BY p.cliente_id ORDER BY p.fecha) as gasto_acumulado
FROM pedidos p
JOIN clientes c ON p.cliente_id = c.id
ORDER BY c.nombre, p.fecha
'''
print('\n3. Gasto acumulado por cliente (SUM OVER):')
print(pd.read_sql(query_acum, conn).to_string(index=False))

Preguntas:

- ¿Cuál es la diferencia entre `RANK()` y `ROW_NUMBER()` cuando hay empates?

- ¿Para qué sirve `PARTITION BY` dentro de una Window Function?

- ¿Cómo usarías `LAG()` para calcular el crecimiento porcentual entre períodos?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Python + SQL: Queries desde Código (15 mins)

En tu trabajo diario como Data Scientist, raramente escribirás SQL en una consola aislada. Lo más común es ejecutar queries **desde Python** y trabajar con los resultados como DataFrames.

### Herramientas de conexión

| Herramienta | Cuándo usarla |
|---|---|
| `sqlite3` | Bases de datos locales (archivos .db) — incluido en Python |
| `pd.read_sql()` | Ejecutar query y obtener DataFrame directamente |
| `SQLAlchemy` | Conexión a cualquier BD (PostgreSQL, MySQL, BigQuery) |
| `psycopg2` | Específico para PostgreSQL |

### Queries parametrizadas (seguridad)

**NUNCA** construyas queries con f-strings o concatenación de strings. Esto te expone a **SQL injection** (un atacante puede ejecutar código malicioso).

```python
# ❌ MAL — vulnerable a SQL injection
query = f"SELECT * FROM clientes WHERE ciudad = '{ciudad}'"

#  BIEN — query parametrizada (segura)
query = "SELECT * FROM clientes WHERE ciudad = ?"
pd.read_sql(query, conn, params=[ciudad])
```

In [ ]:
# === PYTHON + SQL: FLUJO COMPLETO ===

# 1. Query parametrizada
ciudad = 'CDMX'
query_param = 'SELECT nombre, ciudad FROM clientes WHERE ciudad = ?'
resultado = pd.read_sql(query_param, conn, params=[ciudad])
print(f'Clientes en {ciudad}:')
print(resultado)

# 2. Guardar un DataFrame como tabla SQL
df_nuevo = pd.DataFrame({'producto': ['Tablet', 'Monitor'], 'precio': [8000, 12000]})
df_nuevo.to_sql('productos_nuevos', conn, if_exists='replace', index=False)
print('\n✅ DataFrame guardado como tabla SQL')
print(pd.read_sql('SELECT * FROM productos_nuevos', conn))

# 3. Ejecutar query compleja y analizar en Pandas
query_analisis = '''
WITH resumen AS (
    SELECT cliente_id, SUM(total) as gasto, COUNT(*) as pedidos
    FROM pedidos GROUP BY cliente_id
)
SELECT c.nombre, c.ciudad, r.gasto, r.pedidos
FROM clientes c JOIN resumen r ON c.id = r.cliente_id
'''
df_resumen = pd.read_sql(query_analisis, conn)
print('\nAnálisis en Pandas después de SQL:')
print(f'  Gasto promedio: ${df_resumen["gasto"].mean():,.0f}')
print(f'  Cliente top: {df_resumen.loc[df_resumen["gasto"].idxmax(), "nombre"]}')

conn.close()

Preguntas:

- ¿Por qué las queries parametrizadas son más seguras que usar f-strings?

- ¿Cuándo conviene hacer el análisis en SQL vs traer los datos a Pandas?

- ¿Qué ventaja tiene `pd.read_sql()` sobre ejecutar queries manualmente con `cursor.execute()`?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (15 mins)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

### Ejercicio

Usando la base de datos de e-commerce (recréala con el código de arriba):

1. Escribe una query con **CTE** que encuentre los clientes que gastaron más que el promedio general
2. Usa **ROW_NUMBER()** para obtener el pedido más reciente de cada cliente
3. Usa **LAG()** para calcular el cambio porcentual entre pedidos consecutivos de Ana García
4. Crea un **ranking** de productos más vendidos por categoría
5. Conecta los resultados a Pandas y crea una visualización del gasto por ciudad

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: Recrea la conexión para el ejercicio


## <span style="color: #2826DE;">Tips y buenas prácticas

- **Usa CTEs** en lugar de subconsultas anidadas — son más legibles y mantenibles
- **Indexa** las columnas que usas frecuentemente en `WHERE` y `JOIN`
- **Evita `SELECT *`** — selecciona solo las columnas que necesitas
- **Queries parametrizadas** siempre — nunca uses f-strings para construir SQL
- **Comenta tu SQL** con `--` para explicar la lógica de negocio
- **Window Functions > subconsultas** para rankings y comparaciones temporales
- **Prueba cada CTE por separado** antes de combinarlas

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué hace una Window Function que GROUP BY no puede?

- Filtrar filas antes de agrupar
- Calcular agregados sin colapsar las filas originales 
- Crear tablas temporales
- Hacer JOINs entre más de 2 tablas


2️⃣ ¿Qué es una CTE en SQL?

- Un tipo de índice para acelerar queries
- Una tabla temporal definida con WITH al inicio de la query 
- Un JOIN especial para tablas grandes
- Una constraint de integridad referencial


3️⃣ ¿Cómo evitas SQL injection en Python?

- Usando f-strings con validación manual
- Con queries parametrizadas usando ? o %s 
- Con try/except alrededor de la query
- Usando .format() con escape de caracteres


4️⃣ ¿Qué función de Pandas ejecuta una query SQL y devuelve un DataFrame?

- pd.query()
- pd.read_sql() 
- pd.from_sql()
- pd.execute()


5️⃣ ¿Qué devuelve `LAG(total, 1)` en una Window Function?

- El valor de la fila siguiente
- El valor de la fila anterior 
- El primer valor de la partición
- El último valor de la partición


6️⃣ ¿Por qué no puedes usar un alias de SELECT en WHERE?

- Porque WHERE no acepta texto
- Porque WHERE se ejecuta antes que SELECT en el orden de ejecución 
- Porque los alias solo funcionan en ORDER BY
- Porque WHERE solo acepta números

## <span style="color: #2826DE;">Cierre

SQL es una habilidad que usarás todos los días como Data Scientist. Las Window Functions y CTEs son lo que separa a un usuario básico de SQL de un profesional. Practica con HackerRank SQL y construye queries cada vez más complejas. 